# Qwen-Image-2.1 / Viggle Turbo Standalone Runner (Colab / Cloud GPU)

Run **Viggle Qwen-Image-2.1 Turbo (GGUF)** and **Base Qwen-Image-2.1** on Google Colab (Free T4 or A100/L4 GPU) without ComfyUI.

Supports both **Text-to-Image** and **Image Editing** (Image-to-Image) with fully configurable Steps, CFG, Model variants, and Resolutions.

In [ ]:
# 1. Verify NVIDIA GPU
!nvidia-smi

In [ ]:
# 2. Clone Repository & Setup CUDA Engine
!git clone https://github.com/samirpatil2000/vision-model.git
%cd vision-model

# Build stable-diffusion.cpp with CUDA acceleration
!chmod +x setup.sh download_models.sh run_inference.sh edit_image.sh
!./setup.sh

In [ ]:
# 3. Download Model Weights
# Choose variant:
#   --turbo-q4  : 4-step fast Turbo (Q4_K_M, default, recommended for T4)
#   --turbo-q8  : 4-step high-fidelity Turbo (Q8_0, best quality on A100)
#   --base-q4   : 30-50 step full Base model (Q4_K_M)
#   --base-q8   : 30-50 step full Base model (Q8_0)
!./download_models.sh --turbo-q4

In [ ]:
# @title 4. Generate Image (Text-to-Image) { run: "auto" }
prompt = "A raw 35mm photograph of an elderly craftsman carving wood in a sunlit workshop, photorealistic" # @param {type:"string"}
negative_prompt = "" # @param {type:"string"}
steps = 4 # @param {type:"slider", min:1, max:50, step:1}
cfg_scale = 1.0 # @param {type:"slider", min:1.0, max:15.0, step:0.5}
width = 1024 # @param [512, 768, 1024, 1280] {type:"raw"}
height = 1024 # @param [512, 768, 1024, 1280] {type:"raw"}
seed = 42 # @param {type:"integer"}
output_file = "./output/colab_generation.png" # @param {type:"string"}

import os
from IPython.display import Image, display

os.environ["STEPS"] = str(steps)
os.environ["CFG"] = str(cfg_scale)
os.environ["WIDTH"] = str(width)
os.environ["HEIGHT"] = str(height)
os.environ["SEED"] = str(seed)
if negative_prompt:
    os.environ["NEGATIVE_PROMPT"] = negative_prompt
else:
    os.environ.pop("NEGATIVE_PROMPT", None)

!./run_inference.sh "{prompt}" "{output_file}"

if os.path.exists(output_file):
    display(Image(output_file))


In [ ]:
# @title 5. Edit Existing Image (Image-to-Image) { run: "auto" }
input_image = "./output/colab_generation.png" # @param {type:"string"}
edit_prompt = "Replace the wooden block with a chocolate cake, and the saw with a kitchen knife" # @param {type:"string"}
output_edit = "./output/colab_edit.png" # @param {type:"string"}
steps = 4 # @param {type:"slider", min:1, max:50, step:1}
cfg_scale = 1.0 # @param {type:"slider", min:1.0, max:15.0, step:0.5}
width = 1024 # @param [512, 768, 1024, 1280] {type:"raw"}
height = 1024 # @param [512, 768, 1024, 1280] {type:"raw"}

import os
from IPython.display import Image, display

os.environ["STEPS"] = str(steps)
os.environ["CFG"] = str(cfg_scale)
os.environ["WIDTH"] = str(width)
os.environ["HEIGHT"] = str(height)

!./edit_image.sh "{input_image}" "{edit_prompt}" "{output_edit}"

if os.path.exists(output_edit):
    display(Image(output_edit))
